In [16]:
import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"

In [17]:
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels, embed_dim, patch_size=16):
        super().__init__()
        self.conv2d = nn.Conv2d(embed_dim, in_channels,
                                kernel_size=patch_size, stride=patch_size)

    def forward(self, X):
        X = self.conv2d(X)
        X = X.flatten(start_dim=2)
        return X.transpose(1, 2)

In [2]:
class ViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 num_classes=1000, embed_dim=768, depth=12, num_heads=12,
                 ff_dim=3072, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(embed_dim, in_channels, patch_size)
        cls_init = torch.randn(1, 1, embed_dim) * 0.02
        self.cls_token = nn.Parameter(cls_init)
        num_patches = (img_size // patch_size) ** 2
        pos_init = torch.randn(1, num_patches + 1, embed_dim) * 0.02
        self.pos_embed = nn.Parameter(pos_init)
        self.dropout = nn.Dropout(p=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, activation="gelu", batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.output = nn.Linear(embed_dim, num_classes)

    def forward(self, X):
        Z = self.patch_embed(X)
        cls_expd = self.cls_token.expand(Z.shape[0], -1, -1)
        Z = torch.cat((cls_expd, Z), dim=1)
        Z = Z + self.pos_embed
        Z = self.dropout(Z)
        Z = self.encoder(Z)
        Z = self.layer_norm(Z[:, 0])
        logits = self.output(Z)
        return logits

In [3]:
vit_model = ViT(
    img_size=224, patch_size=16, in_channels=3, num_classes=1000, embed_dim=768,
    depth=12, num_heads=12, ff_dim=3072, dropout=0.1)
batch = torch.randn(4, 3, 224, 224)
logits = vit_model(batch)
logits.shape

torch.Size([4, 1000])

In [9]:
from datasets import load_dataset

pets = load_dataset("timm/oxford-iiit-pet")

In [10]:
from transformers import ViTForImageClassification, AutoImageProcessor

model_id = "google/vit-base-patch16-224-in21k"
vit_model = ViTForImageClassification.from_pretrained(model_id, num_labels=37).to(device)
vit_processor = AutoImageProcessor.from_pretrained(model_id, use_fast=True)

Loading weights:   0%|          | 0/6 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
encoder.layer.{0...11}.attention.attention.query.weight | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.key.weight   | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.value.weight | UNEXPECTED | 
encoder.layer.{0...11}.intermediate.dense.weight        | UNEXPECTED | 
encoder.layer.{0...11}.intermediate.dense.bias          | UNEXPECTED | 
pooler.dense.bias                                       | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.query.bias   | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_before.bias            | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.bias                | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.key.bias     | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_after.bias

In [11]:
def vit_collate_fn(batch):
    images = [example["image"] for example in batch]
    labels = [example["label"] for example in batch]
    inputs = vit_processor(images, return_tensors="pt", do_convert_rgb=True)
    inputs["labels"] = torch.tensor(labels)
    return inputs


In [19]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments("my_pets_vit", per_device_train_batch_size=32, eval_strategy="epoch",
                         num_train_epochs=3, remove_unused_columns=False)
trainer = Trainer(model=vit_model, args=args, data_collator=vit_collate_fn,
                  train_dataset=pets["train"], eval_dataset=pets["test"])
train_output = trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,3.438522
2,No log,3.320237
3,No log,3.275694


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [1]:
from transformers import pipeline

model_id = "openai/clip-vit-base-patch32"
clip_pipeline = pipeline(task="zero-shot-image-classification", model=model_id,
                         device_map="auto", dtype="auto")
candidate_labels = ["cricket", "ladybug", "spider"]
image_url = "https://homl.info/ladybug"
results = clip_pipeline(image_url, candidate_labels=candidate_labels,
                        hypothesis_template="This is a photo of a {}.")

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

In [2]:
results

[{'score': 0.9972797632217407, 'label': 'ladybug'},
 {'score': 0.0016558406641706824, 'label': 'spider'},
 {'score': 0.0010643524583429098, 'label': 'cricket'}]

In [4]:
import PIL
import torch
import urllib.request
from transformers import CLIPProcessor, CLIPModel

clip_processor = CLIPProcessor.from_pretrained(model_id)
clip_model = CLIPModel.from_pretrained(model_id)
image = PIL.Image.open(urllib.request.urlopen(image_url)).convert("RGB")
captions = [f"This is a photo of a {label}." for label in candidate_labels]
inputs = clip_processor(text=captions, images=[image], return_tensors="pt",
                        padding=True)
with torch.no_grad():
    outputs = clip_model(**inputs)

text_features = outputs.text_embeds    # shape [3, 512]  # 3 captions
image_features = outputs.image_embeds  # shape [1, 512]  # 1 image (ladybug)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [5]:
similarities = image_features @ text_features.T  # shape [1, 3]
similarities

tensor([[0.2336, 0.3021, 0.2380]])

In [6]:
temperature = clip_model.logit_scale.detach().exp()
rescaled_similarities = similarities * temperature
probabilities = torch.nn.functional.softmax(rescaled_similarities , dim=1)
probabilities

tensor([[0.0011, 0.9973, 0.0017]])

In [ ]:
# from transformers import Blip2Processor, Blip2ForConditionalGeneration
# import torch
# device = "cuda" if torch.cuda.is_available() else "cpu"
#
# model_id = "Salesforce/blip2-opt-2.7b"
# blip2_processor = Blip2Processor.from_pretrained(model_id)
# blip2_model = Blip2ForConditionalGeneration.from_pretrained(
#     model_id, device_map=device, dtype=torch.float16)
#
# image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
# image = Image.open(urllib.request.urlopen(image_url))
# inputs = blip2_processor(images=image, return_tensors="pt")
# inputs = inputs.to(device, dtype=torch.float16)
# with torch.no_grad():
#     generated_ids = blip2_model.generate(**inputs)
#
# generated_text = blip2_processor.batch_decode(generated_ids)
# print(generated_text)
# generated_text = blip2_processor.batch_decode(generated_ids, skip_special_tokens=True)
# print(generated_text)